In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F
import psycopg2


spark = SparkSession.builder \
    .appName("Spark SCD2 Pipeline") \
    .config("spark.sql.shuffle.partitions", "6") \
    .config("spark.streaming.kafka.maxRatePerPartition", "10000") \
    .config(
        "spark.jars.packages",
        "org.postgresql:postgresql:42.7.3"
    ) \
    .getOrCreate()


spark.conf.set("spark.sql.shuffle.partitions", "4")

spark



In [ ]:
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "broker2:29094") \
    .option("subscribe", "cdc.cdc_db.customers") \
    .option("group.id", "consumer-type-group-test-1") \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", "false") \
    .load()

value_schema = StructType([
    StructField("payload", StructType([
        StructField("before", StructType([
            StructField("customer_id", IntegerType()),
            StructField("full_name", StringType()),
            StructField("email", StringType()),
            StructField("status", StringType())
        ])),
        StructField("after", StructType([
            StructField("customer_id", IntegerType()),
            StructField("full_name", StringType()),
            StructField("email", StringType()),
            StructField("status", StringType())
        ])),
        StructField("op", StringType()),
        StructField("ts_ms", LongType())
    ]))
])

parsed_df = kafka_df.select(
    F.from_json(F.col("value").cast("string"), value_schema).alias("data")
)

customers_df = parsed_df.select(
    F.col("data.payload.after.customer_id").alias("customer_id"),
    F.col("data.payload.after.full_name").alias("full_name"),
    F.col("data.payload.after.email").alias("email"),
    F.col("data.payload.after.status").alias("status"),
    F.col("data.payload.op").alias("op"),
    F.col("data.payload.ts_ms").alias("event_ts")
).filter("customer_id IS NOT NULL")
postgres_url = "jdbc:postgresql://postgresDB:5432/ECOMMERCE"
postgres_properties = {
    "user": "admin",
    "password": "admin",
    "driver": "org.postgresql.Driver"
}

TARGET_COLUMNS = [
    "customer_id",
    "full_name",
    "email",
    "status",
    "row_hash",
    "start_date",
    "end_date",
    "is_current",
    "ingestion_ts"
]


def scd2_upsert(batch_df, batch_id):

    if batch_df.isEmpty():
        return

    batch_df = batch_df.withColumn(
        "row_hash",
        md5(concat_ws("||", "full_name", "email", "status"))
    )

    # Load current records ONCE
    current_df = (
        spark.read.jdbc(
            url=postgres_url,
            table="customers_history",
            properties=postgres_properties
        )
        .filter("is_current = true")
    )

    changes_df = (
        batch_df.alias("i")
        .join(current_df.alias("c"), "customer_id", "left")
        .filter(
            F.col("c.customer_id").isNull() |
            (F.col("i.row_hash") != F.col("c.row_hash"))
        )
        .select(
            "i.customer_id",
            "i.full_name",
            "i.email",
            "i.status",
            "i.row_hash"
        )
    )

    if changes_df.isEmpty():
        return

    ids_to_close = [
        row.customer_id
        for row in changes_df.select("customer_id").distinct().collect()
    ]

    if ids_to_close:
        import psycopg2

        conn = psycopg2.connect(
            host="postgresDB",
            database="ECOMMERCE",
            user="admin",
            password="admin"
        )
        cur = conn.cursor()

        cur.execute(
            """
            UPDATE customers_history
            SET is_current = false,
                end_date = NOW()
            WHERE customer_id = ANY(%s)
              AND is_current = true
            """,
            (ids_to_close,)
        )

        conn.commit()
        cur.close()
        conn.close()


    inserts_df = (
        changes_df
        .withColumn("start_date", F.current_timestamp())
        .withColumn("end_date", F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True))
        .withColumn("ingestion_ts", F.current_timestamp())
    )

    inserts_df.write.jdbc(
        url=postgres_url,
        table="customers_history",
        mode="append",
        properties=postgres_properties
    )



main_query = customers_df.writeStream \
    .foreachBatch(scd2_upsert) \
    .outputMode("update") \
    .option("checkpointLocation", "/tmp/checkpoints/customers_history") \
    .start()

main_query.awaitTermination()
